# Chapter 1 PR #69 × continuous-trait local reanalysis

## tl;dr

- 27 continuous endpoints are explicitly defined; 17 have frozen numerical measurements locally and 10 remain genuinely unexecuted.
- The local ≤10 km high-resolution diagnostic screen ran 48 endpoint–climate models after a CSV round-trip bug was fixed. Four unadjusted candidate rows passed the tier-wise BH screen, but all are screening-only and none survives the locked PR #69 image-quality/stratum gate.
- All 72 locked PR #69 involucre model rows were reproduced to numerical tolerance, and 0/3 canonical endpoints passed the frozen resolution/sharpness retention rule.
- The submission-facing biological conclusion therefore remains the two small primary rows retained by PR #69: orientation–BIO1 and outline aspect ratio–BIO4. The local 17-endpoint run is a diagnostic bridge, not a replacement FDR family.

## Context & Methods

This notebook joins two frozen streams without changing their scientific roles. The 3,725-observation balanced image atlas supplies already measured continuous orientation, colour/display and outline values. The PR #69 high-resolution table supplies three executed contour formulas, mapped to their sole canonical endpoint identifiers. Missing endpoints remain missing.

For diagnostics, all available primary and candidate endpoints are joined to the CHELSA values present for the 1,292 high-resolution observations. Linear responses use taxon-demeaned standardized slopes with taxon-clustered uncertainty; hue is one joint circular test per predictor. BH correction is separate for primary and candidate tiers. This diagnostic subset lacks the full 46,276-observation seasonal, dominant-taxon and native-range control structure, so it cannot create or replace submission claims.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

ROOT = Path.cwd()
assert (ROOT / "manuscript" / "final_claims.json").exists(), "Run from the repository root"
LOCAL = ROOT / "analysis_outputs" / "local_pr69_continuous_reanalysis"

with (ROOT / "manuscript" / "final_claims.json").open(encoding="utf-8") as handle:
    final_claims = json.load(handle)
with (LOCAL / "universe_17" / "continuous_trait_universe_report.json").open(encoding="utf-8") as handle:
    universe_report = json.load(handle)
with (LOCAL / "involucre_bridge" / "bridge_report.json").open(encoding="utf-8") as handle:
    bridge_report = json.load(handle)
with (LOCAL / "diagnostic_climate_screen_17_le10km" / "continuous_trait_universe_climate_report.json").open(encoding="utf-8") as handle:
    screen_report = json.load(handle)

traits = pd.read_csv(LOCAL / "universe_17" / "continuous_trait_universe_observation_long.csv", low_memory=False)
coverage = pd.read_csv(LOCAL / "diagnostic_climate_screen_17_le10km" / "continuous_trait_universe_coverage.csv")
coefficients = pd.read_csv(LOCAL / "diagnostic_climate_screen_17_le10km" / "continuous_trait_universe_climate_coefficients.csv")
gate = pd.read_csv(LOCAL / "involucre_bridge" / "canonical_involucre_headline_audit.csv")
gate_comparison = pd.read_csv(LOCAL / "reviewer_gate_comparison" / "candidate_screen_vs_pr69_gates.csv")
contract = pd.read_csv(ROOT / "ch1_global" / "v2" / "ontology" / "ch1_continuous_trait_contract.csv", keep_default_na=False)

## Data

The hundreds of thousands of photographs were not reduced through one arbitrary convenience sample. There are two analysis streams. The exhaustive stream supports the primary within-taxon climate analysis after coordinate and spatial thinning; the smaller balanced atlas supports repeated head-level measurement, variance decomposition and development/audit. The local bridge uses the latter because the 923.7 MB exhaustive artifact is not available inside this local connector session.

In [2]:
datasets = final_claims["datasets"]
attrition = pd.DataFrame([
    {"stream": "exhaustive", "stage": "detector-positive observations", "n_observations": 406_582, "n_taxa": 286, "role": "post-detection source"},
    {"stream": "exhaustive", "stage": "finite coordinates", "n_observations": 392_989, "n_taxa": 271, "role": "coordinate QC"},
    {"stream": "exhaustive", "stage": "positional accuracy ≤10 km", "n_observations": 297_293, "n_taxa": 259, "role": "geographic accuracy QC"},
    {"stream": "exhaustive", "stage": "taxon × 0.25° spatial thinning", "n_observations": 46_276, "n_taxa": 259, "role": "primary climate cohort"},
    {"stream": "balanced atlas", "stage": "one photograph per observation", "n_observations": 3_725, "n_taxa": 216, "role": "measurement/variance/historical audit"},
    {"stream": "high-resolution", "stage": "usable involucre measurements", "n_observations": 1_292, "n_taxa": 210, "role": "candidate contour traits"},
    {"stream": "high-resolution", "stage": "≤10 km complete cases", "n_observations": 904, "n_taxa": 165, "role": "locked PR69 bias-control family"},
])
attrition

,stream,stage,n_observations,n_taxa,role
0,exhaustive,detector-positive observations,406582,286,post-detection source
1,exhaustive,finite coordinates,392989,271,coordinate QC
2,exhaustive,positional accuracy ≤10 km,297293,259,geographic accuracy QC
3,exhaustive,taxon × 0.25° spatial thinning,46276,259,primary climate cohort
4,balanced atlas,one photograph per observation,3725,216,measurement/variance/historical audit
5,high-resolution,usable involucre measurements,1292,210,candidate contour traits
6,high-resolution,≤10 km complete cases,904,165,locked PR69 bias-control family


In [3]:
assert traits.duplicated(["obs_id", "endpoint_id"]).sum() == 0
assert set(traits["endpoint_id"]).issubset(set(contract["endpoint_id"]))
assert universe_report["category_columns_in_output"] == []
assert bridge_report["n_involucre_outside_primary"] == 0
assert bridge_report["n_taxon_name_mismatches"] == 0
assert bridge_report["model_reproduction"]["status"] == "exact_within_tolerance"

endpoint_quality = (
    traits.groupby(["endpoint_id", "module", "analysis_tier"], dropna=False)
    .agg(
        n_rows=("obs_id", "size"),
        n_observations_measured=("measurement_available", "sum"),
        n_taxa_measured=("taxon_name", lambda s: s[traits.loc[s.index, "measurement_available"].astype(bool)].nunique()),
    )
    .reset_index()
)
endpoint_quality["measurement_coverage"] = endpoint_quality["n_observations_measured"] / endpoint_quality["n_rows"]
endpoint_quality.sort_values(["analysis_tier", "module", "endpoint_id"])

,endpoint_id,module,analysis_tier,n_rows,n_observations_measured,n_taxa_measured,measurement_coverage
0,bract_projection_maximum,armature,candidate,3725,1292,210,0.346846
16,visible_floret_fraction,display,candidate,3725,3327,215,0.893154
1,bract_projection_roughness,involucre_architecture,candidate,3725,1292,210,0.346846
2,bract_spread_fraction,involucre_architecture,candidate,3725,1292,210,0.346846
11,corolla_purple_pixel_fraction,colour,descriptive_only,3725,3327,215,0.893154
12,corolla_redmagenta_pixel_fraction,colour,descriptive_only,3725,3327,215,0.893154
13,corolla_white_pixel_fraction,colour,descriptive_only,3725,3327,215,0.893154
14,corolla_yellow_pixel_fraction,colour,descriptive_only,3725,3327,215,0.893154
7,corolla_hue_cos,colour,primary,3725,3325,215,0.892617
8,corolla_hue_sin,colour,primary,3725,3325,215,0.892617


In [4]:
quality_summary = pd.DataFrame([
    {"check": "contract endpoints", "value": len(contract), "status": "pass"},
    {"check": "executed endpoints in local universe", "value": universe_report["n_available_endpoints"], "status": "partial by design"},
    {"check": "unexecuted endpoints", "value": universe_report["n_missing_unexecuted_endpoints"], "status": "requires image remeasurement"},
    {"check": "duplicate obs_id × endpoint_id", "value": int(traits.duplicated(["obs_id", "endpoint_id"]).sum()), "status": "pass"},
    {"check": "category columns leaked", "value": len(universe_report["category_columns_in_output"]), "status": "pass"},
    {"check": "PR69 coefficient rows reproduced", "value": bridge_report["model_reproduction"]["n_compared_rows"], "status": "pass"},
    {"check": "successful diagnostic models", "value": screen_report["n_successful_endpoint_predictor_models"], "status": "diagnostic only"},
])
quality_summary

,check,value,status
0,contract endpoints,27,pass
1,executed endpoints in local universe,17,partial by design
2,unexecuted endpoints,10,requires image remeasurement
3,duplicate obs_id × endpoint_id,0,pass
4,category columns leaked,0,pass
5,PR69 coefficient rows reproduced,72,pass
6,successful diagnostic models,48,diagnostic only


## Results

The diagnostic 17-endpoint universe contains all currently available numerical measurements and no biological categories. Four descriptive floral-pixel composition fractions are retained as numerical description but are not fitted as four independent ecological outcomes. Ten endpoints remain absent because their required high-resolution image functions have not been executed on the frozen exhaustive cohort.

In [5]:
model_overview = pd.DataFrame([
    {"metric": "linear endpoints modelled", "value": screen_report["n_linear_endpoints_modelled"]},
    {"metric": "joint circular traits modelled", "value": screen_report["n_circular_traits_modelled"]},
    {"metric": "successful endpoint–predictor models", "value": screen_report["n_successful_endpoint_predictor_models"]},
    {"metric": "primary BH signals", "value": screen_report["n_primary_fdr_signals"]},
    {"metric": "candidate BH signals", "value": screen_report["n_candidate_fdr_signals"]},
])
model_overview

,metric,value
0,linear endpoints modelled,11
1,joint circular traits modelled,1
2,successful endpoint–predictor models,48
3,primary BH signals,0
4,candidate BH signals,4


In [6]:
display_columns = [
    "endpoint_id", "analysis_tier", "predictor", "inferential_unit",
    "beta_std_within", "effect_magnitude", "p_value", "q_fdr_bh_within_tier",
    "n_observations", "n_taxa",
]
top_models = coefficients.sort_values("q_fdr_bh_within_tier").head(12)
top_models[[column for column in display_columns if column in top_models.columns]]

,endpoint_id,analysis_tier,predictor,inferential_unit,beta_std_within,effect_magnitude,p_value,q_fdr_bh_within_tier,n_observations,n_taxa
41,bract_spread_fraction,candidate,env_chelsa_bio04_native,linear_endpoint,0.102551,NaN,0.001677,0.016640,904,165
33,bract_projection_roughness,candidate,env_chelsa_bio04_native,linear_endpoint,0.102457,NaN,0.002080,0.016640,904,165
37,bract_projection_maximum,candidate,env_chelsa_bio04_native,linear_endpoint,0.083539,NaN,0.006348,0.033856,904,165
34,bract_projection_roughness,candidate,env_chelsa_bio12_native,linear_endpoint,-0.071639,NaN,0.009288,0.037152,904,165
5,corolla_lab_lightness,primary,env_chelsa_bio04_native,linear_endpoint,0.103075,NaN,0.004727,0.151271,779,155
42,bract_spread_fraction,candidate,env_chelsa_bio12_native,linear_endpoint,-0.064225,NaN,0.061394,0.196460,904,165
15,visible_floret_fraction,candidate,env_chelsa_bio15_native,linear_endpoint,0.066477,NaN,0.095812,0.255500,779,155
38,bract_projection_maximum,candidate,env_chelsa_bio12_native,linear_endpoint,-0.036339,NaN,0.262654,0.600351,904,165
20,capitulum_outline_circularity,primary,env_chelsa_bio01_native,linear_endpoint,-0.037731,NaN,0.319176,0.729545,731,150
27,capitulum_outline_solidity,primary,env_chelsa_bio15_native,linear_endpoint,-0.041620,NaN,0.304268,0.729545,731,150


In [7]:
gate_comparison[[
    "endpoint_id", "predictor", "screen_beta", "screen_q",
    "quality_adjusted_beta", "quality_adjusted_q",
    "positive_in_all_successful_strata", "failure_reasons",
    "submission_claim_eligible",
]]

,endpoint_id,predictor,screen_beta,screen_q,quality_adjusted_beta,quality_adjusted_q,positive_in_all_successful_strata,failure_reasons,submission_claim_eligible
0,bract_projection_roughness,env_chelsa_bio04_native,0.102457,0.016640,0.086646,0.073008,False,fails_locked_quality_adjusted_bh;fails_resolut...,False
1,bract_spread_fraction,env_chelsa_bio04_native,0.102551,0.016640,0.094176,0.069580,True,fails_locked_quality_adjusted_bh;independent_b...,False
2,bract_projection_maximum,env_chelsa_bio04_native,0.083539,0.033856,0.078308,0.073008,False,fails_locked_quality_adjusted_bh;fails_resolut...,False
3,bract_projection_roughness,env_chelsa_bio12_native,-0.071639,0.037152,-0.058983,0.188799,False,fails_locked_quality_adjusted_bh;independent_b...,False


In [8]:
gate_display = gate[[
    "endpoint", "adjusted_bio4_beta", "adjusted_bio4_q",
    "positive_in_all_successful_strata", "strict_resolution_control_retained",
    "claim_status", "submission_eligible",
]].copy()
gate_display

,endpoint,adjusted_bio4_beta,adjusted_bio4_q,positive_in_all_successful_strata,strict_resolution_control_retained,claim_status,submission_eligible
0,bract_projection_roughness,0.086646,0.073008,False,False,withdrawn_by_pr69_resolution_gate,False
1,bract_spread_fraction,0.094176,0.069580,True,False,withdrawn_by_pr69_resolution_gate,False
2,bract_projection_maximum,0.078308,0.073008,False,False,withdrawn_by_pr69_resolution_gate,False


In [9]:
native_rows = pd.DataFrame(
    final_claims["bias_control_reanalysis"]["native_range_sensitivity"]["rows"]
)
native_rows[[
    "endpoint", "predictor", "native_only_beta", "native_only_q",
    "n_observations", "n_taxa", "native_range_robust",
]]

,endpoint,predictor,native_only_beta,native_only_q,n_observations,n_taxa,native_range_robust
0,orientation_angle,BIO1,0.022648,3.105900e-02,21725,128,True
1,corolla_chroma,BIO12,0.027812,7.538700e-02,23672,128,False
2,shape_aspect_ratio,BIO4,0.028420,1.247176e-17,22528,126,True
3,shape_aspect_ratio,BIO12,-0.015666,1.331810e-01,22528,126,False


## Takeaways

1. **The current submission conclusion does not change.** PR #69 retains orientation–BIO1 and outline aspect ratio–BIO4 only; it withdraws all three high-resolution involucre/BIO4 rows.
2. **The continuous-trait redesign is now testable but incomplete.** Seventeen endpoints are in one category-free long table. Six candidate architecture/armature endpoints and four validation-only surface endpoints still require image execution; missingness is explicit.
3. **The “hundreds of thousands versus a few thousand” issue is a cohort-ledger issue, not simple data waste.** The 46,276-observation exhaustive primary cohort is the main climate-analysis stream. The 3,725-observation atlas and 1,292-observation involucre subset serve different measurement and validation purposes and must not share FDR labels.
4. **A small q value in the generic screen is not a result to publish.** Four candidate rows have q < 0.05 before endpoint-specific quality controls; the locked PR #69 reanalysis reduces the corresponding involucre headline to 0/3 retained endpoints. The diagnostic subset also lacks the full seasonal, dominant-taxon, native-range and developmental-stage controls.
5. **Next executable gate:** obtain the frozen exhaustive artifact in an environment without the 512 MB connector cap, run all 27 endpoint functions, and then pass every promoted endpoint through its matching PR #69 bias/validation gate.